# 03 · Model training

| Stage | Notebook |
|---|---|
| 1. Data quality & EDA | `01_eda.ipynb` |
| 2. Feature engineering | `02_feature_engineering.ipynb` |
| **3. Model training** | **this notebook** |
| 4. Evaluation | `04_evaluation.ipynb` |

This notebook **recomputes** training through `aerocast.pipeline`, the same functions `scripts/run_phase1.py` calls. **Every table shown here is asserted equal to the artefact the script saved in `results/`** (`artefacts.assert_matches`), so the notebook cannot drift from the script, which remains the single source of truth for the numbers. Runtime is about 3 minutes.

Selection in this notebook uses **training-window evidence only** (rolling-origin CV). Calibration data and test data are not touched here. See the development protocol in notebook 04, §0.

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().resolve().parent / "src"))  # make `aerocast` importable
import warnings; warnings.filterwarnings("ignore")
import json
import numpy as np, pandas as pd
from IPython.display import display
from aerocast import artefacts, config, models, pipeline, plots
pd.set_option("display.precision", 4); pd.set_option("display.width", 160)
R = config.RESULTS_DIR
S = json.loads((R / "summary.json").read_text())

## 1. Chronological split

In [2]:
P = pipeline.prepare()
splits = pd.DataFrame(P.split_summary()).T
pd.testing.assert_frame_equal(splits, pd.DataFrame(S["splits"]).T)   # == summary.json
splits.assign(role=["fitting + rolling-origin CV", "calibration + threshold choice", "touched once, final numbers"])

,rows,delay_rate,from,to,role
train,211684,0.2649,2013-01-08,2013-08-31,fitting + rolling-origin CV
calib,55628,0.157,2013-09-01,2013-10-31,calibration + threshold choice
test,53991,0.2583,2013-11-01,2013-12-31,"touched once, final numbers"


* **Jan 1–7 excluded** (warm-up): history windows are empty or very thin.
* **Base-rate shift:** 26.5% in training, 15.7% in calibration, 25.8% in test (Nov 18.1%, **Dec 33.5%**).

## 2. The two models: theory in brief

**Logistic regression.** It models $P(y=1\mid x) = \sigma(w^\top x + b)$ and minimises the **L2-regularised log-loss**

$$\mathcal{L}(w) = -\frac1n\sum_i \big[y_i\log p_i + (1-y_i)\log(1-p_i)\big] + \frac{1}{2Cn}\lVert w\rVert^2 .$$

The penalty shrinks coefficients toward 0. That's needed here because the history features are strongly correlated (notebook 01, §8). Smaller *C* means stronger shrinkage. The loss is convex, so there is a single optimum.

**XGBoost.** An additive ensemble of trees $F_T(x)=\sum_t f_t(x)$, fitted stage-wise. At stage *t* it minimises a **second-order Taylor approximation** of the loss plus a complexity penalty:

$$\tilde{\mathcal L}^{(t)} = \sum_i \big[g_i f_t(x_i) + \tfrac12 h_i f_t(x_i)^2\big] + \gamma T_{\text{leaves}} + \tfrac12\lambda\lVert w\rVert^2,$$

with $g_i, h_i$ the first and second derivatives of the log-loss at the current prediction ($g_i = p_i - y_i$, $h_i = p_i(1-p_i)$). For a fixed tree structure, the optimal leaf weight is $w_j^* = -G_j/(H_j+\lambda)$, where $G_j, H_j$ are the leaf's sums of gradients and Hessians. The split gain uses the same quantities. Regularisation comes from λ (`reg_lambda`), `min_child_weight` (minimum Hessian mass per leaf), depth, shrinkage (`learning_rate`), row/column subsampling, and the number of trees.

**Why log-loss?** It's a **strictly proper scoring rule**: its expected value is minimised only by reporting the true probability. A model trained on log-loss is therefore pushed toward calibrated probabilities on its training distribution. (The Brier score is also proper. Accuracy and AUC are not; AUC only measures ranking.)

## 3. Baselines (reference rows next to every learned model)

* **Prior (base rate)**: the training base rate for every flight (PR-AUC = the evaluated window's base rate, ROC-AUC = 0.5);
* **Route history**: the smoothed 14-day carrier-route delay rate on its own, no model.

In [3]:
raw_preds = pipeline.baseline_preds(P)
base_scores = pipeline.mean_scores(raw_preds)
print(artefacts.assert_matches(base_scores, "mean_scores.csv", key="model"))
base_scores.set_index("model").add_prefix("mean score · ")

OK: identical to results/mean_scores.csv


,mean score · train,mean score · calib,mean score · test
model,,,
prior,0.2649,0.2649,0.2649
route_history,0.2650,0.1574,0.2420


## 4. Why rolling-origin CV instead of one validation month

The first design tuned on **September** alone. September is the calmest month of the year (13.7% late), and its scores didn't track later months. The diagnostic below fits XGBoost (final hyperparameters) on Jan–Aug and plots PR-AUC on September and on October as trees are added. It uses the calibration window only, never test.

In [4]:
curve = pipeline.single_month_tree_curve(P, S["xgb_params"])
print(artefacts.assert_matches(curve, "single_month_tree_curve.csv"))
fig = plots.tree_count_curve(curve, S["xgb_params"]["n_estimators"])
plots.save(fig, "train_single_month_tree_curve"); fig

OK: identical to results/single_month_tree_curve.csv


<Figure size 650x320 with 1 Axes>

In [5]:
pd.DataFrame({m: {"best n_trees": int(curve.loc[curve[c].idxmax(), "n_trees"]), "best PR-AUC": curve[c].max(),
                   "PR-AUC range": curve[c].max() - curve[c].min()} for m, c in [("Sep", "pr_auc_m9"), ("Oct", "pr_auc_m10")]}).T

,best n_trees,best PR-AUC,PR-AUC range
Sep,30.0,0.2359,0.0445
Oct,50.0,0.3009,0.0283


Two adjacent months give different "best" tree counts, on nearly flat curves, so a single month is a noisy selector. **Rolling-origin CV** inside the training window replaces it:

| Fold | Train | Validate |
|---|---|---|
| 1 | Jan–Apr | May |
| 2 | Jan–May | Jun |
| 3 | Jan–Jun | Jul |
| 4 | Jan–Jul | Aug |

Training never sees a month at or after the one it's validated on (asserted in `tests/`).

## 5. Logistic regression: C by rolling CV

In [6]:
lr = pipeline.train_logreg(P)
print(artefacts.assert_matches(lr.cv.reset_index(drop=True), "cv_logreg.csv"))
wide = lr.cv.pivot(index="C", columns="fold", values="val_pr_auc")
wide.assign(mean=wide.mean(axis=1), std=wide.std(axis=1))

OK: identical to results/cv_logreg.csv


fold,0,1,2,3,mean,std
C,,,,,,
0.0003,0.4268,0.4997,0.5440,0.3935,0.4660,0.0683
0.0010,0.4308,0.5016,0.5481,0.3933,0.4684,0.0695
0.0030,0.4338,0.5025,0.5498,0.3930,0.4698,0.0699
0.0100,0.4353,0.5030,0.5509,0.3928,0.4705,0.0702
0.1000,0.4354,0.5029,0.5515,0.3922,0.4705,0.0706
1.0000,0.4354,0.5028,0.5514,0.3917,0.4703,0.0708


In [7]:
print("chosen C =", lr.C)
fig = plots.cv_logreg_curve(lr.cv); plots.save(fig, "train_cv_logreg"); fig

chosen C = 0.1


<Figure size 600x320 with 1 Axes>

The curve is flat above C ≈ 0.003. C = 0.1 is the (marginal) maximum and lies inside the grid. The fold-to-fold spread (Aug ≈ 0.39 vs Jul ≈ 0.55) is far larger than any difference between C values, and it tracks each month's base rate: PR-AUC rises with the share of late flights.

*Joint tuning with the history settings.* The smoothing strength m and the long window were first scored at a fixed C = 0.001. Notebook 02, §6 re-tunes (m, window) **jointly with C** (each pair scored at its own best C): the same pair wins and C = 0.1 again, so the earlier choice stands.

## 6. XGBoost: configuration search

**Grid** (`models.XGB_GRID`, 48 combinations, 16 sampled at random):
* depth {2, 3, 5}: the order of interactions;
* min_child_weight {20, 100}: minimum Hessian mass per leaf, which matters for a weak, noisy signal;
* colsample {0.5, 0.8} and λ {1, 10}: decorrelation and leaf shrinkage;
* learning rate {0.03, 0.1}: shrinkage, traded off against the number of trees, which early stopping sets.

For each configuration and fold: up to 3,000 trees with **early stopping** (100 rounds) on that fold's validation month. Score = mean fold PR-AUC; tree count = mean of the four best iterations.

In [8]:
xgb = pipeline.train_xgb(P, n_configs=16)
print(artefacts.assert_matches(xgb.cv_configs.reset_index(), "cv_xgb_configs.csv"))
xgb.cv_configs.head(8)

OK: identical to results/cv_xgb_configs.csv


,mean_pr_auc,std_pr_auc,n_trees
config,,,
0,0.4754,0.0673,143
15,0.4725,0.0741,146
1,0.4711,0.0694,340
4,0.4708,0.0764,76
5,0.4704,0.0688,231
10,0.4692,0.0684,314
11,0.4692,0.0685,374
14,0.4683,0.0683,64


In [9]:
assert xgb.params == S["xgb_params"]
pd.Series({**xgb.params, "cv_mean_pr_auc (early-stopped)": xgb.cv_configs.iloc[0]["mean_pr_auc"],
           "cv_std_pr_auc": xgb.cv_configs.iloc[0]["std_pr_auc"]}, name="chosen").to_frame()

,chosen
max_depth,2.0000
learning_rate,0.1000
min_child_weight,100.0000
subsample,0.8000
colsample_bytree,0.8000
reg_lambda,10.0000
n_estimators,143.0000
cv_mean_pr_auc (early-stopped),0.4754
cv_std_pr_auc,0.0673


The best configuration has **depth-2 trees** (pairwise interactions at most), min_child_weight = 100 and λ = 10.0, so it's heavily constrained. The top configurations are separated by less than one fold standard deviation (about 0.07).

## 7. Which model is final? (rule fixed before test)

**Rule:** the model with the higher mean rolling-CV PR-AUC, **each scored as deployed**.

The XGBoost search score early-stops on each **validation** fold. That means the tree count is chosen on the very data it's scored on, which makes the score **optimistic**. The deployed XGBoost uses a *fixed* tree count (the fold average). So it's re-scored on the same folds with that fixed count, and that's the number used for selection. LogReg has no in-fold tuning, so its CV score is already "as deployed". The baselines are scored on the same folds.

In [10]:
cv = pipeline.cv_summary(lr, xgb, P)
cv_tbl = pipeline.cv_table(cv)
print(artefacts.assert_matches(cv_tbl, "cv_summary.csv"))
cv_tbl.drop(columns="variant")

OK: identical to results/cv_summary.csv


,model,calibration,cv_pr_auc_mean,cv_pr_auc_std,fold_m5,fold_m6,fold_m7,fold_m8
0,Prior (base rate),none,0.2775,0.0483,0.2267,0.3152,0.3221,0.2460
1,Route history,none,0.3476,0.0601,0.2766,0.3911,0.4034,0.3194
2,LogReg (as deployed),raw,0.4705,0.0706,0.4354,0.5029,0.5515,0.3922
3,"XGBoost (as deployed, fixed trees)",raw,0.4450,0.0729,0.4044,0.4284,0.5523,0.3950
4,"XGBoost (in-fold early stopping, optimistic; n...",raw,0.4754,0.0673,0.4490,0.4956,0.5574,0.3995


In [11]:
final = pipeline.select_final_model(cv)
print("final model:", final, "| config.FINAL_MODEL:", config.FINAL_MODEL)
pair = pipeline.paired_cv_difference(cv)
assert pair == S["cv_paired_logreg_minus_xgboost"]
pd.Series({"per-fold LogReg − XGBoost (May..Aug)": np.round(pair["diffs"], 4).tolist(),
           "mean difference": pair["mean"], "sd of difference": pair["std"],
           "folds LogReg better": f'{pair["folds_a_better"]}/{pair["n_folds"]}'}, name="paired").to_frame()

final model: logreg | config.FINAL_MODEL: logreg


,paired
per-fold LogReg − XGBoost (May..Aug),"[0.031, 0.0746, -0.0007, -0.0027]"
mean difference,0.0255
sd of difference,0.0362
folds LogReg better,2/4


### What the evidence does and does not show

* As deployed, **LogReg 0.4705 vs XGBoost 0.4450** (fold sd about 0.07 for both). In-fold early stopping would inflate XGBoost to 0.4754, which would have reversed the decision.
* **The two models are not statistically distinguished.** The mean gap (0.0255) is much smaller than the fold-to-fold sd (about 0.07), and the test bootstrap CI of XGBoost − LogReg in notebook 04 includes 0. The paired fold-by-fold differences (table above) are less affected by the month effect than the raw sd suggests, but four folds are far too few to call a winner. We say only: LogReg is **not worse** than XGBoost, in CV or on test.
* **The final-model choice therefore rests on the rule declared before test** (higher CV mean as deployed, recorded in `config.FINAL_MODEL`, and the pipeline fails if CV ever disagrees), not on a demonstrated superiority. The simpler model also wins ties on interpretability and cost.
* This choice uses training-window CV only. The development protocol (notebook 04, §0) records that an intermediate version of this comparison used the optimistic score.

## 8. Final models (fit on all of Jan–Aug)

In [12]:
raw_preds["logreg"] = pipeline.predict_splits(lr.model, P)
raw_preds["xgboost"] = pipeline.predict_splits(xgb.model, P)
scores = pipeline.mean_scores(raw_preds)
print(artefacts.assert_matches(scores, "mean_scores.csv"))
scores.set_index("model").add_prefix("mean score · ")

OK: identical to results/mean_scores.csv


,mean score · train,mean score · calib,mean score · test
model,,,
prior,0.2649,0.2649,0.2649
route_history,0.2650,0.1574,0.2420
logreg,0.2649,0.2300,0.2581
xgboost,0.2649,0.2029,0.2317


The raw mean score on Sep–Oct (0.23 for LogReg, 0.20 for XGBoost) is well above that window's 15.7% base rate, because the models carry the training base rate (26.5%). Calibration (notebook 04) decides whether to correct this.

## 9. Consistency with `scripts/run_phase1.py`

In [13]:
assert lr.C == S["logreg_C"] and xgb.params == S["xgb_params"]
assert final == S["final_model"]["model"]
assert all(np.allclose(cv[m]["folds"], S["cv_pr_auc"][m]["folds"]) for m in cv)

_, saved_raw = pipeline.load_predictions()
for m in pipeline.RAW_SCORES:
    for s in config.SPLITS:
        assert np.array_equal(raw_preds[m][s].astype(saved_raw[m][s].dtype), saved_raw[m][s]), (m, s)
print("OK: every table above, the hyper-parameters, the final-model choice and all predictions match results/")

OK: every table above, the hyper-parameters, the final-model choice and all predictions match results/
